## Introducción

La tesoreria general de la republica entrega los datos que consolidan las patentes por no uso de derechos de aprovechamiento de aguas en Chile.
Este conjunto de datos permite clasificar las tesorerías regionales con más patentes por no uso de aprovechamiento de aguas además de permitir clasificar si los ddeudores son 
personas naturales o personas Juridicas.

**Objetivo general (Fase 2).** Construir un *pipeline* reproducible que deje el *dataset*
limpio, codificado y escalado, listo para el modelado posterior.

**Objetivos específicos.**
- Obtener y explorar los datos verificando su estructura y calidad.
- Limpiar gestionando rigurosamente los valores nulos.
- Transformar las variables categóricas nominales con codificación One-Hot.
- Estandarizar las variables continuas.
- Validar técnicamente el resultado.

**Atributos:** `id`, `gender`, `age`, `hypertension`, `heart_disease`, `ever_married`,
`work_type`, `Residence_type`, `avg_glucose_level`, `bmi`, `smoking_status` y la variable
objetivo `stroke` (1 = sufrió accidente cerebrovascular, 0 = no).

### Librerías utilizadas

| Librería | Para qué la usamos |
|---|---|
| `numpy` | Cálculo numérico y manejo de arreglos. |
| `pandas` | Cargar y manipular la tabla de datos (el `DataFrame`). |
| `matplotlib` | Generar los gráficos (*boxplots*). |
| `sklearn.preprocessing` | Las herramientas de codificación (`LabelEncoder`, `OneHotEncoder`) y escalamiento (`StandardScaler`). |

`np.random.seed(42)` fija la semilla aleatoria: garantiza que cualquier proceso con azar
dé **siempre el mismo resultado**, lo que asegura la *reproducibilidad* exigida en la fase.

In [ ]:
import sys                                   # intérprete del kernel
import subprocess                            # instalar paquetes faltantes
import importlib                             # refrescar el sistema de importación
import importlib.util                        # comprobar si un paquete está instalado

# Verificación del entorno: instala en el intérprete del kernel lo que falte.
REQUERIDOS = {                               # módulo: nombre del paquete en pip
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",               # se importa como sklearn, se instala como scikit-learn
}
faltantes = [pkg for mod, pkg in REQUERIDOS.items()
             if importlib.util.find_spec(mod) is None]

if faltantes:
    print("Instalando:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
    importlib.invalidate_caches()

import numpy as np                           # cálculo numérico y operaciones vectorizadas
import pandas as pd                          # estructuras tabulares: Series y DataFrame
import matplotlib.pyplot as plt              # gráficos de control
from sklearn import preprocessing            # LabelEncoder y OneHotEncoder
from sklearn.preprocessing import StandardScaler   # estandarización z-score

# Reproducibilidad: fijamos la semilla aleatoria del entorno
np.random.seed(42)

%matplotlib inline

In [ ]:
from pathlib import Path      # manejo de rutas independiente del sistema operativo
import sys

# sys.version trae la versión completa con fecha de compilación;
# split()[0] deja solo el número, que es lo único que hay que comprobar.
print("Python:", sys.version.split()[0])
for lib, mod in [("numpy", np), ("pandas", pd)]:
    print(f"{lib:8}:", mod.__version__)

# Estructura del proyecto. parents=True crea las carpetas intermedias;
# exist_ok=True evita el error si ya existen.
def localizar_raiz(inicio=None):
    """
    Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde está guardado el cuaderno, no en la
    carpeta desde la que se lanzó JupyterLab. Como cada cuaderno vive en la carpeta
    de su fase (F1, F2, F3 o F4), si la carpeta de trabajo es una de ellas la raíz
    es la carpeta superior. Si el cuaderno se ejecuta desde la raíz, se usa tal cual.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()                 # la misma función del cuaderno de la Fase 1
DIR_CRUDO = RAIZ / "data" / "raw"            # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # resultado del pipeline
DIR_DOCS = RAIZ / "docs"                     # diccionario, bitácora y metadatos
for carpeta in (DIR_CRUDO, DIR_PROCESADO, DIR_DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO = DIR_CRUDO / "tb_patentes_aguas_december_2025_data.csv"
print("\nArchivo esperado en:", ARCHIVO)

### Procedencia del conjunto de datos

| Campo | Valor |
| --- | --- |
| Título | consolidacion de patentes por no uso de derechos de aprovechamiento de aguas |
| Autor | Tesoreía General de La República |
| Plataforma | Datos.gob.cl |
| Enlace | `https://datos.gob.cl/dataset/tb-patentes-aguas/resource/4d1d615f-5935-4760-b399-1dd76b281715` |
| Estructura esperada | 46.468 filas × 21 columnas originales· 34 nulos en `tesoreria_regional` |
| Unidad de observación | Pago patentes de agua|

**Referencia en APA 7 para el informe:**

> **Referencia en APA 7 para el informe:**
Tesorería General de la República. (2025). Reporte consolidado de recaudación y cartera morosa de patentes de aguas (diciembre 2025) [Conjunto de datos]. Ministerio de Hacienda, Gobierno de Chile. https://datos.gob.cl/dataset/tb-patentes-aguas/resource/4d1d615f-5935-4760-b399-1dd76b281715

Documentar la procedencia no es un trámite: sin ella, nadie puede verificar sobre
qué versión del archivo se trabajó.

## 1. Obtención de los datos

In [ ]:
def cargar_datos(ruta):
    """
    Carga un conjunto de datos desde un archivo CSV.

    Parámetros
    ----------
    ruta : str o Path
        Ruta del archivo CSV.

    Retorna
    -------
    pd.DataFrame
        Datos cargados en un DataFrame.

    Lanza
    -----
    FileNotFoundError
        Si el archivo no existe, con un mensaje que indica dónde debe estar.
    """
    ruta = Path(ruta)
    try:
        # En este archivo los faltantes de bmi vienen como el texto 'N/A':
        # na_values los convierte en NaN al leer.
        df = pd.read_csv(ruta, na_values=["N/A"])
    except FileNotFoundError:
        # from None oculta la traza interna de pandas: queda solo el mensaje útil.
        raise FileNotFoundError(
            f"No se encontró {ruta.name} en {ruta.parent.resolve()}.\n"
            "Descárguelo desde el enlace de la tabla de procedencia y guárdelo en "
            "data/raw/ de la raíz del proyecto."
        ) from None
    print(f"Datos cargados: {df.shape[0]} filas y {df.shape[1]} columnas.")
    return df


df_crudo = cargar_datos(ARCHIVO)

# Se conserva el original sin modificaciones y se trabaja sobre una copia
df = df_crudo.copy()
df.head()


## 2. Exploración inicial

**Qué hace este paso.** Antes de modificar nada, miramos el estado original de los datos.
Esto nos da la "línea base" contra la cual verificaremos después cada transformación.

In [ ]:
def explorar_dataframe(df):
    """Resumen exploratorio: dimensiones, tipos, nulos y estadisticos descriptivos."""
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isnull().sum())
    print("\nEstadisticos descriptivos (variables numericas):")
    return df.describe()


explorar_dataframe(df)

In [ ]:
# Conteo de categorias en las variables nominales (control de calidad de entrada)
columnas_categoricas = ['CUENTA', 'FORM_FOLIO', 'SALDO',
                        'PAGADO', 'MOV_TIPO']
for col in columnas_categoricas:
    print(f"\n{col}:")
    # dropna=False incluye los nulos como una categoría más: si una variable
    # tiene faltantes queremos verlos aquí, no que desaparezcan del conteo.
    print(df[col].value_counts(dropna=False))